## 介紹 

本課程將涵蓋： 
- 什麼是函式呼叫及其使用案例 
- 如何使用 Azure OpenAI 建立函式呼叫 
- 如何將函式呼叫整合到應用程式中 

## 學習目標 

完成本課程後，您將了解並會如何： 

- 使用函式呼叫的目的 
- 使用 Azure OpenAI 服務設定函式呼叫 
- 針對應用案例設計有效的函式呼叫 


## 了解函式呼叫

在本課程中，我們想為我們的教育新創公司建構一個功能，讓使用者能使用聊天機器人來尋找技術課程。我們將根據他們的技能等級、目前角色和感興趣的技術來推薦課程。

要完成這項功能，我們會結合使用：
 - `Azure Open AI` 為使用者創建聊天體驗
 - `Microsoft Learn Catalog API` 幫助使用者根據其需求找到課程
 - `函式呼叫` 將使用者的查詢傳送到函式以進行 API 請求。

開始之前，讓我們先來看看為什麼我們會想使用函式呼叫：


### 為什麼要使用函式調用 

如果你已完成本課程中的其他任何課程，你大概已了解使用大型語言模型（LLMs）的強大功能。希望你也能看到它們的一些限制。 

函式調用是 Azure Open AI 服務的一項功能，用來克服以下限制： 
1) 回應格式一致性 
2) 能夠在聊天情境中使用應用程式其他來源的資料 

在函式調用出現之前，LLM 的回應是無結構且不一致的。開發人員需要撰寫複雜的驗證程式碼，確保能處理回應的各種變化。 

使用者無法得到像「斯德哥爾摩現在的天氣如何？」這類的答案。這是因為模型的資料侷限於訓練時的時間點。 

讓我們看看下面的範例來說明這個問題： 

假設我們想建立一個學生資料庫，以便能為他們建議適合的課程。下面有兩個描述學生的例子，它們在資料內容上非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想將此內容傳送給大型語言模型 (LLM) 來解析資料。這稍後可以用在我們的應用程式中，將資料傳送到 API 或儲存到資料庫。 

讓我們建立兩個相同的提示，指示 LLM 我們感興趣的資訊內容： 


我們想將這個發送給大型語言模型（LLM），以解析對我們產品重要的部分。因此，我們可以創建兩個相同的提示來指導LLM： 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


在建立這兩個提示後，我們將使用 `client.responses.create` 將它們發送給 LLM。我們將提示儲存在 `input` 變數中，並將角色指定為 `user`。這是為了模擬使用者傳送訊息給聊天機器人。



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

現在我們可以將兩個請求發送給 LLM，並檢查我們收到的回應。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



儘管提示相同且描述相似，我們仍可能獲得不同格式的 `Grades` 屬性。 

如果你多次執行上述儲存格，格式可能是 `3.7` 或 `3.7 GPA`。 

這是因為大型語言模型（LLM）接收的是非結構化的書面提示資料，並且回傳的也是非結構化資料。我們需要有結構化的格式，才能在儲存或使用這些資料時知道會得到什麼。 

透過使用函式調用，我們可以確保收到結構化資料。使用函式調用時，LLM 並不會真正呼叫或執行任何函式，而是我們為 LLM 建立一個回應需要遵守的結構。接著，我們根據這些結構化回應決定在應用程式中該執行什麼函式。  
 


![函式呼叫流程圖](../../../../translated_images/zh-TW/Function-Flow.083875364af4f4bb.webp)


接著，我們可以將從函式返回的結果傳回給 LLM。然後 LLM 會使用自然語言來回應，用以回答使用者的查詢。


### 使用函式呼叫的應用案例 

<strong>呼叫外部工具</strong> 
聊天機器人非常擅長回答使用者的問題。透過使用函式呼叫，聊天機器人可以利用使用者的訊息來完成特定任務。例如，學生可以請聊天機器人「發電子郵件給我的講師，說我需要更多關於這個主題的協助」。此時可以呼叫 `send_email(to: string, body: string)` 函式。


**建立 API 或資料庫查詢**
使用者可以用自然語言查找資訊，並將其轉換成格式化的查詢或 API 請求。舉例來說，一位老師可能會詢問「哪些學生完成了最後一次作業？」這就可以呼叫名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函式。


<strong>建立結構化資料</strong>
使用者可以從一段文字或 CSV 檔案中，利用大型語言模型提取重要資訊。例如，學生可以將維基百科上有關和平協議的文章轉換為 AI 問題卡。這可以透過呼叫名為 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 的函式來完成。


## 2. 建立您的第一個函式呼叫

建立函式呼叫的過程包含三個主要步驟：
1. 使用函式列表和使用者訊息呼叫 Chat Completions API
2. 讀取模型的回應以執行動作，例如執行函式或 API 呼叫
3. 再次呼叫 Chat Completions API，並帶入函式的回應，以使用該資訊建立回應給使用者。


![函式呼叫流程](../../../../translated_images/zh-TW/LLM-Flow.3285ed8caf4796d7.webp)


### 函式呼叫的元素

#### 使用者輸入

第一步是建立使用者訊息。這可以透過取得文字輸入的值來動態指定，或者你也可以在此處指定一個值。如果你是第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。

`role` 可以是 `system`（建立規則）、`assistant`（模型）或 `user`（最終使用者）。對於函式呼叫，我們會指定為 `user` 並給出一個範例問題。


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 建立函式。 

接下來我們將定義一個函式及該函式的參數。我們這裡只使用一個函式，叫做 `search_courses`，但你可以建立多個函式。

<strong>重要</strong> ：函式會包含在系統訊息中傳給大型語言模型（LLM），並會計入你可用的 token 數量中。 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong>

`name` - 想要被調用的函式名稱。

`description` - 描述函式的運作方式。在這裡具體且清楚非常重要。

`parameters` - 您希望模型在回應中產生的值和格式列表。


`type` - 屬性將被儲存的資料類型。

`properties` - 模型將用於回應的具體值列表。


`name` - 模型在格式化回應中會使用的屬性名稱。

`type` - 該屬性的資料類型。

`description` - 具體屬性的描述。


<strong>選用</strong>

`required` - 完成函式呼叫所需的必填屬性。


### 呼叫函式
定義函式後，我們現在需要在呼叫 Chat Completion API 時包含它。我們透過在請求中加入 `functions` 來做到這一點。在這個例子中，`functions=functions`。

也有一個選項可以將 `function_call` 設為 `auto`。這表示我們會讓大型語言模型 (LLM) 根據使用者訊息決定應該呼叫哪個函式，而不是由我們自己指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在讓我們來看看回應內容，以及它如何被格式化：

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函式的名稱被呼叫了，並且從使用者訊息中，LLM 能夠找到合適函式參數的資料。


## 3.將函式呼叫整合到應用程式中。


在我們測試過來自 LLM 格式化的回應後，現在可以將其整合到應用程式中。

### 管理流程

要將此整合到我們的應用程式中，讓我們採取以下步驟：

首先，讓我們呼叫 Open AI 服務，並將訊息存到名為 `response_message` 的變數中。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函式，用來呼叫 Microsoft Learn API 以取得課程清單： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實踐，我們接著會查看模型是否想要調用一個函式。之後，我們將建立其中一個可用的函式並將其與被調用的函式相匹配。 
接著，我們會取得函式的參數並將它們映射到 LLM 的參數上。

最後，我們會附加函式調用訊息和 `search_courses` 訊息返回的值。這樣就能提供 LLM 所需的所有資訊，
以自然語言回應使用者。 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將把更新的訊息發送給 LLM，以便我們可以收到自然語言的回應，而不是 API JSON 格式的回應。


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 代碼挑戰 

做得好！若要繼續學習 Azure Open AI 函數呼叫，你可以構建： https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 更多函數參數，可能幫助學習者找到更多課程。你可以在這裡找到可用的 API 參數： 
 - 創建另一個函數呼叫，從學習者那裡獲取更多資訊，例如他們的母語 
 - 在函數呼叫和／或 API 呼叫未返回任何合適課程時，建立錯誤處理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
